# Lesson 7.3 - Context Engineering, Memory & Planning (toy agent demo)

## Objectives

- Build session memory and long-term retrieval.
- Implement context compression via summarization.
- Execute a simple plan and re-plan loop.


In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## Knowledge Base and Session Data


In [2]:
DOCS = [
    "Customer ACME uses region ap-south and prefers email communication.",
    "ACME had outage incident on June 2 and requested root-cause summary format.",
    "Support SLA for enterprise tier is 2 hours for P1 incidents.",
    "Billing disputes over $10,000 must include finance approval before closure.",
]

SESSION_TURNS = [
    "User asked for status update on ACME onboarding.",
    "Agent responded with timeline and asked for blocker details.",
    "User reported security questionnaire still pending.",
]


## Long-Term Memory Retrieval (TF-IDF demo)

In production, this is often vector DB retrieval. Here we keep a local, lightweight approximation.


In [3]:
vectorizer = TfidfVectorizer()
kb_matrix = vectorizer.fit_transform(DOCS)


def retrieve(query: str, top_k: int = 2) -> List[str]:
    q = vectorizer.transform([query])
    sims = cosine_similarity(q, kb_matrix).flatten()
    idx = np.argsort(-sims)[:top_k]
    return [DOCS[i] for i in idx]


retrieve("What are ACME SLA obligations and blockers?")


['Customer ACME uses region ap-south and prefers email communication.',
 'ACME had outage incident on June 2 and requested root-cause summary format.']

## Session Memory Compression

We compress prior turns into a concise summary so context stays within budget.


In [4]:
def summarize_turns(turns: List[str], max_sentences: int = 2) -> str:
    selected = turns[-max_sentences:]
    return " ".join(selected)


session_summary = summarize_turns(SESSION_TURNS)
session_summary


'Agent responded with timeline and asked for blocker details. User reported security questionnaire still pending.'

## Planning and Re-Planning

Plan format is explicit so each step can be validated.


In [5]:
@dataclass
class PlanStep:
    step_id: int
    action: str
    status: str = "pending"


@dataclass
class PlannerState:
    goal: str
    steps: List[PlanStep] = field(default_factory=list)


def build_initial_plan(goal: str) -> PlannerState:
    steps = [
        PlanStep(1, "retrieve_relevant_context"),
        PlanStep(2, "draft_status_report"),
        PlanStep(3, "check_policy_constraints"),
    ]
    return PlannerState(goal=goal, steps=steps)


def maybe_replan(state: PlannerState, missing_info: bool) -> None:
    if missing_info:
        state.steps.insert(1, PlanStep(99, "request_missing_details"))


planner_state = build_initial_plan("Prepare ACME onboarding report")
maybe_replan(planner_state, missing_info=True)
[(s.step_id, s.action) for s in planner_state.steps]


[(1, 'retrieve_relevant_context'),
 (99, 'request_missing_details'),
 (2, 'draft_status_report'),
 (3, 'check_policy_constraints')]

## End-to-End Toy Agent Run


In [6]:
def run_agent(goal: str, query: str) -> Dict[str, str]:
    plan = build_initial_plan(goal)
    retrieved = retrieve(query, top_k=2)
    summary = summarize_turns(SESSION_TURNS)

    missing = "SLA" not in " ".join(retrieved)
    maybe_replan(plan, missing_info=missing)

    report = {
        "goal": goal,
        "session_summary": summary,
        "retrieved_context": " | ".join(retrieved),
        "plan": " -> ".join([s.action for s in plan.steps]),
    }
    return report


output = run_agent(
    goal="Prepare customer-ready onboarding status report",
    query="ACME SLA and pending blockers",
)
output


{'goal': 'Prepare customer-ready onboarding status report',
 'session_summary': 'Agent responded with timeline and asked for blocker details. User reported security questionnaire still pending.',
 'retrieved_context': 'Customer ACME uses region ap-south and prefers email communication. | ACME had outage incident on June 2 and requested root-cause summary format.',
 'plan': 'retrieve_relevant_context -> request_missing_details -> draft_status_report -> check_policy_constraints'}

## Connect to Theory

- Retrieval = long-term semantic memory.
- Session summary = short-term memory compression.
- Explicit plan objects = inspectable planning policy.
- Re-planning trigger = reliability mechanism for missing context.


## Business Case Studies & Exceptions

### Case Study A: CRM Copilot Memory Design

A CRM copilot combined conversation summaries with account-level semantic memory. This reduced repeated questions and improved continuity between account managers.

### Case Study B: Planning Failure in Travel Automation

A travel assistant failed when policy constraints changed during booking. Re-planning on retrieval mismatch reduced incorrect bookings.

### Exceptions

- Highly regulated environments may prohibit persistent user memory.
- Aggressive summarization can drop critical details and increase error rates.


## Interview Questions & Answers

1. **Q: What is context engineering?**
   **A:** Selecting and structuring the right context for each model call.
2. **Q: Why not keep full chat history forever?**
   **A:** Token limits, cost, and increased noise.
3. **Q: What is short-term memory in agent systems?**
   **A:** Session-local state for current task continuity.
4. **Q: What is long-term memory?**
   **A:** Persistent knowledge across sessions, often via retrieval stores.
5. **Q: How does summarization help?**
   **A:** It compresses state while preserving key facts.
6. **Q: What is a planning object?**
   **A:** A structured representation of ordered tasks and statuses.
7. **Q: When should re-planning occur?**
   **A:** On tool failure, missing evidence, or changed constraints.
8. **Q: How do you evaluate retrieval quality?**
   **A:** Precision@k, recall@k, and relevance judgments.
9. **Q: How can context hurt performance?**
   **A:** Irrelevant or stale context can mislead the model.
10. **Q: What production control is most important?**
   **A:** Explicit context budget + observability on retrieval and summaries.
